# Mechanical P-Bit

In [1]:
import jax
from jax import lax
import jax.numpy as jnp
import jax.random as jr
import diffrax
import matplotlib.pyplot as plt
from functools import partial

from diffrax import ControlTerm, MultiTerm, ODETerm, PIDController

jax.config.update("jax_enable_x64", True)

## 1. Define Parameters

In [2]:
num_oscillators = 2

mass_arr = jnp.array([1.0 for _ in range(num_oscillators)])
mass_output = 0.01

w_0 = 1.0
period = 2*jnp.pi/w_0
w_0_arr = jnp.array([w_0 for _ in range(num_oscillators)])
w_0_output = w_0

w_p = 2*w_0

F_p = 0.05
Qual = 100.0
Qual_arr = jnp.array([Qual for _ in range(num_oscillators)])
Qual_output = Qual

gamma = 0.003

k = 1e-3
k_arr = jnp.array([k for _ in range(num_oscillators)])

k_B = 1.0
T = 0.2

# Compute noise strength    
noise_strength =  jnp.sqrt(4*w_0*T/(2*Qual*mass_output))  
noise_strength_output =  jnp.sqrt(4*w_0_output*T/(2*Qual_output*mass_output))  

# Simulation Parameters
t0 = 0.0  
num_periods = 2000# Start time
t1 = num_periods*period
dt0 = period/1000
num_pts_per_period = 100
ts_dim = jnp.linspace(t0, t1, num_periods*num_pts_per_period)

system_size = 2 * (num_oscillators+1)

# Initial Conditions
y0 = jnp.zeros(system_size)

## 2. Define drift and diffusion functions for SDE

In [3]:
# 2.1. Direct Integration SDE for x(t)
def drift_x_fn(t, state, args):
    """
    Drift function for the direct integration of x(t).
    y = [x, v]
    """
    
    x_output = state[-2]
    
        
    #solve for dx_i/dt, dv_i/dt
    
    def calculate_derivatives(i):
        x_curr = state[2*i]
        v_curr = state[2*i+1]
        w_0 = w_0_arr[i]
        qual_fac = Qual_arr[i]
        mass = mass_arr[i]
        k_curr = k_arr[i]
        
        dx_curr = v_curr
        dv_curr = -w_0/(2 * qual_fac * mass) * v_curr - (w_0**2 - F_p/mass * jnp.cos(w_p * t)) * x_curr - gamma/mass * x_curr**3 - k_curr/mass * (x_curr - x_output)
        return dx_curr, dv_curr
    
    derivative_array = jax.vmap(calculate_derivatives)(jnp.arange(num_oscillators))
    dx_dv_arr = jnp.concatenate([derivative_array[0], derivative_array[1], jnp.zeros(2)])
        
    #solve for dx_0/dt, dv_0/dt
    v_output = state[-1]
    dx_output_dt = v_output
    
    
    def calculate_spring_acc(i):
        return k_arr[i]**3/mass_output * (state[2*i] - x_output)
    
    spring_acc = jnp.sum(jax.vmap(calculate_spring_acc)(jnp.arange(num_oscillators)))
        
    dv_output_dt = -w_0_output/(2 * Qual_output * mass_output) * v_output - (w_0_output**2 - 0.*F_p/mass_output * jnp.cos(w_p * t)) * x_output + spring_acc
    
    dx_dv_arr.at[-2].set(dx_output_dt)
    dx_dv_arr.at[-1].set(dv_output_dt)
    
    
    return dx_dv_arr

In [4]:
def diffusion_x_fn(t, state, args):
    """
    Diffusion 
    """
        
    noise_matrix = jnp.zeros((system_size, system_size))
    
    def update_noise_matrix(i, noise_matrix):
        return noise_matrix.at[2 * i + 1, 2 * i + 1].set(noise_strength)

    final_noise_matrix = lax.fori_loop(0, num_oscillators, update_noise_matrix, noise_matrix)
    
    noise_matrix = noise_matrix.at[-1, -1].set(noise_strength_output)
    
    
    return noise_matrix

## 3. Solve SDE

In [5]:
w_shape = (system_size,)  # state is (x1, p1)
brownian_motion = diffrax.VirtualBrownianTree(
    t0, t1, 1.e-11, w_shape, jr.PRNGKey(0), diffrax.SpaceTimeLevyArea
)
terms_x = MultiTerm(ODETerm(drift_x_fn), ControlTerm(diffusion_x_fn, brownian_motion))
saveat = diffrax.SaveAt(ts=ts_dim)


In [ ]:

# sample 1000 points
solution_x = diffrax.diffeqsolve(
    terms_x,
    solver=diffrax.SRA1(),
    t0=t0,
    t1=t1,
    dt0=dt0,
    y0=y0,
    args=(),
    saveat=saveat,
    progress_meter=diffrax.TqdmProgressMeter(),
    max_steps=100000000000,
    # stepsize_controller=PIDController(rtol=rtol, atol=atol),
)


/Users/sowmyathanvantri/opt/anaconda3/lib/python3.9/site-packages/jax/_src/core.py:678: FutureWarning: unhashable type: <class 'jax._src.interpreters.partial_eval.DynamicJaxprTracer'>. Attempting to hash a tracer will lead to an error in a future JAX release.
  warnings.warn(
/Users/sowmyathanvantri/opt/anaconda3/lib/python3.9/site-packages/jax/_src/core.py:678: FutureWarning: unhashable type: <class 'jax._src.interpreters.partial_eval.DynamicJaxprTracer'>. Attempting to hash a tracer will lead to an error in a future JAX release.
  warnings.warn(
/Users/sowmyathanvantri/opt/anaconda3/lib/python3.9/site-packages/jax/_src/core.py:678: FutureWarning: unhashable type: <class 'jax._src.interpreters.partial_eval.DynamicJaxprTracer'>. Attempting to hash a tracer will lead to an error in a future JAX release.
  warnings.warn(
/Users/sowmyathanvantri/opt/anaconda3/lib/python3.9/site-packages/jax/_src/core.py:678: FutureWarning: unhashable type: <class 'jax._src.interpreters.partial_eval.Dynami

## 4.  Extract relative phases of hidden oscillators and envelope of output oscillator

In [ ]:
from scipy.signal import hilbert
import numpy as np

x1 = solution_x.ys[:, 0]
x2 = solution_x.ys[:, 2]
x3 = solution_x.ys[:, 4]

# Extract phases using Hilbert transform
analytic_signal_x1 = hilbert(x1)
analytic_signal_x2 = hilbert(x2)
analytic_signal_x3 = hilbert(x3)

# Calculate phase difference between x₁ and x₂
phase_diff = np.angle(analytic_signal_x1) - np.angle(analytic_signal_x2)
# Wrap to [-π, π]
phase_diff = np.mod(phase_diff + np.pi, 2 * np.pi) - np.pi

# Calculate envelope of x₃
envelope = np.abs(analytic_signal_x3)



## 5. Plot results

In [ ]:
# Define start and end times as fractions of total time
start_frac = 0.0  # Start at beginning
end_frac = 1    # Show first 20%

# Calculate indices for the time window
start_idx = int(len(ts_dim) * start_frac)
end_idx = int(len(ts_dim) * end_frac)


# Create a 3x1 figure
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

title = (f'Hidden: Q={Qual}, T={T}, γ={gamma}, Fp={F_p}, ω₀={w_0}, ωp={w_p}\n'
         f'Output: Q={Qual_output}, M={M_output}, ω₀={w_0_output}, k13={k_13}, k23={k_23}')

# ... rest of plotting code ...
fig.suptitle(title, y=1.01)  # y=1.02 moves the title slightly above the figure

# First subplot: x1 and x2
ax1.plot(ts_dim[start_idx:end_idx], solution_x.ys[start_idx:end_idx, 0], label='x₁(t)')
ax1.plot(ts_dim[start_idx:end_idx], solution_x.ys[start_idx:end_idx, 2], label='x₂(t)')
ax1.set_ylabel('x₁, x₂')
ax1.grid(True)
ax1.legend()

# Second subplot: x3
ax2.plot(ts_dim[start_idx:end_idx], solution_x.ys[start_idx:end_idx, 4], label='x₃(t)')
ax2.set_ylabel('x₃')
ax2.grid(True)
ax2.legend()

# Third subplot: Phase difference and envelope
ax3_twin = ax3.twinx()

# Plot phase difference
l1 = ax3.plot(ts_dim[start_idx:end_idx], 
              jnp.abs(phase_diff[start_idx:end_idx]), 
              label='φ₁ - φ₂', color='blue')
ax3.set_ylabel('Absolute Phase Difference (rad)', color='blue')
ax3.tick_params(axis='y', labelcolor='blue')

# Plot envelope
l2 = ax3_twin.plot(ts_dim[start_idx:end_idx], 
                   envelope[start_idx:end_idx],
                   label='x₃ Envelope', color='red', alpha=0.7)
ax3_twin.set_ylabel('Amplitude Envelope', color='red')
ax3_twin.tick_params(axis='y', labelcolor='red')

# Add legend to third subplot
lns = l1 + l2
labs = [l.get_label() for l in lns]
ax3.legend(lns, labs, loc='upper right')

ax3.grid(True, alpha=0.3)
ax3.set_xlabel('Time')



# Create a safe filename by replacing unsafe characters
safe_title = f"pbit_Q{Qual}_T{T}_g{gamma}_Fp{F_p}_w0{w_0}_wp{w_p}_Qins{Qual_output}_Mins{M_output}_w0ins{w_0_output}_k13{k_13}_k23{k_23}"
safe_title = safe_title.replace('.', 'p').replace('-', 'm')  # Replace decimals and minus signs
save_path = f"../out/p-bit_new/{safe_title}.png"  # Add .png extension

# Save and show the figure
# plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()